# Documents – Test notebook (direct function calls)

Tests the **document ingestion** logic by calling the real functions (no HTTP).  
Run from **project root** or ensure the first cell adds the project root to `sys.path`.

## Setup: add project root and load env

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Project root = parent of notebooks/ when cwd is notebooks, else cwd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

print("Project root:", ROOT)

## 1. List documents – `list_document_names()`
Ensures index exists, then returns unique document names from the vector store.

In [ ]:
from services.ingestion import create_index
from vector_store.store import list_document_names

create_index()
names = list_document_names()
print("Documents in store:", names)

## 2. Upload (index) a PDF – `process_and_index_document(text, document_name)`
Extract text from a PDF file, then call the async ingestion function.  
Set `PDF_PATH` to a real PDF path.

In [ ]:
import asyncio
import io
from pathlib import Path

from PyPDF2 import PdfReader
from services.ingestion import create_index, process_and_index_document
from core.config import settings

PDF_PATH = ROOT / "uploaded_files" / "sample.pdf"  # Change to your PDF path

def extract_text_from_pdf(path):
    with open(path, "rb") as f:
        reader = PdfReader(io.BytesIO(f.read()))
    return "".join(p.extract_text() or "" for p in reader.pages)

if not PDF_PATH.is_file():
    print(f"File not found: {PDF_PATH}")
else:
    create_index()
    text = extract_text_from_pdf(PDF_PATH)
    if not text.strip():
        print("No text extracted from PDF.")
    else:
        count, errors = asyncio.run(process_and_index_document(text, PDF_PATH.name))
        print(f"Chunks indexed: {count}, errors: {errors}")
        # Optionally copy file to uploaded_files to mirror API behavior
        settings.upload_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(PDF_PATH, settings.upload_dir / PDF_PATH.name)
        print(f"File copied to {settings.upload_dir / PDF_PATH.name}")

## 3. Delete a document – `delete_documents_by_document_name(name)`
Removes all chunks for that document from FAISS. Optionally remove the file from `uploaded_files/`.

In [ ]:
from services.ingestion import delete_documents_by_document_name
from core.config import settings

DOCUMENT_NAME = "sample.pdf"  # Must match a name from list_document_names()

result = delete_documents_by_document_name(DOCUMENT_NAME)
print("Delete result:", result)

file_path = settings.upload_dir / DOCUMENT_NAME
if file_path.exists():
    file_path.unlink()
    print(f"Removed file: {file_path}")

## 4. Full flow: list → upload → list → delete → list

In [ ]:
import asyncio
import io
import shutil
from PyPDF2 import PdfReader
from services.ingestion import create_index, process_and_index_document, delete_documents_by_document_name
from vector_store.store import list_document_names
from core.config import settings

path = ROOT / "uploaded_files" / "sample.pdf"  # Use your PDF path
if not path.is_file():
    print("Skipping: PDF not found at", path)
else:
    create_index()
    print("1. List (before):", list_document_names())
    with open(path, "rb") as f:
        text = "".join(p.extract_text() or "" for p in PdfReader(io.BytesIO(f.read())).pages)
    count, _ = asyncio.run(process_and_index_document(text, path.name))
    print("2. Upload: chunks_indexed =", count)
    settings.upload_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(path, settings.upload_dir / path.name)
    print("3. List (after upload):", list_document_names())
    out = delete_documents_by_document_name(path.name)
    print("4. Delete:", out)
    (settings.upload_dir / path.name).unlink(missing_ok=True)
    print("5. List (after delete):", list_document_names())